# Week 6 — Improving the CNN (STARTER)
### The Computer Vision Internship | VisionAI Lagos

## Part 1 — Improved CNN with Regularisation

> 🧠 **What Will This Output?**
> If Dropout(0.5) zeroes 50% of neurons during training, the effective capacity is halved. Will the regularised training F1 be higher or lower than Week 5? Should it be?

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self,num_classes=4,dropout_rate=0.5):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(inplace=True),
            nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(inplace=True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(inplace=True),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(inplace=True),nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),
            nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(inplace=True),nn.MaxPool2d(2),nn.Dropout2d(0.15),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(inplace=True),nn.MaxPool2d(2),
        )
        self.gap=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(nn.Dropout(dropout_rate),nn.Linear(256,128),
                                       nn.ReLU(inplace=True),nn.Dropout(dropout_rate*0.6),nn.Linear(128,num_classes))
    def forward(self,x): return self.classifier(self.gap(self.features(x)).flatten(1))

model=ImprovedCNN(4,dropout_rate=0.5).to(DEVICE)
print(f"ImprovedCNN params: {sum(p.numel() for p in model.parameters()):,}")

## Part 2 — Experiment Log

> ⏸️ **Recall Prompt**
> Tunde: change one hyperparameter at a time. In what order should you change dropout, weight decay, and learning rate schedule? If you changed all three at once and saw improvement, what would you NOT know?

In [ ]:
# Fill in val_f1 and test_f1 as you run each experiment
# RULE: change ONE hyperparameter at a time. Record everything.
experiments = {
    "Baseline (Week 5)":   {"dropout":0.4,"weight_decay":1e-4,"scheduler":"ReduceLROnPlateau","val_f1":None,"test_f1":None},
    "Higher Dropout":      {"dropout":0.5,"weight_decay":1e-4,"scheduler":"ReduceLROnPlateau","val_f1":None,"test_f1":None},
    "Higher Weight Decay": {"dropout":0.4,"weight_decay":5e-4,"scheduler":"ReduceLROnPlateau","val_f1":None,"test_f1":None},
    "Cosine Annealing":    {"dropout":0.5,"weight_decay":5e-4,"scheduler":"CosineAnnealingLR","val_f1":None,"test_f1":None},
}
print(f"{'Experiment':<28} {'Dropout':>8} {'WD':>9} {'Val F1':>8} {'Test F1':>9}")
print("-"*60)
for name,p in experiments.items():
    f1_v=f"{p['val_f1']:.3f}" if p['val_f1'] else "TBD"
    f1_t=f"{p['test_f1']:.3f}" if p['test_f1'] else "TBD"
    print(f"  {name:<26} {p['dropout']:>8.1f} {p['weight_decay']:>9.0e} {f1_v:>8} {f1_t:>9}")

In [ ]:
# Train ImprovedCNN with CosineAnnealingLR
# YOUR CODE: fill in the training loop below (same pattern as Week 5)

model_w6 = ImprovedCNN(num_classes=4, dropout_rate=0.5).to(DEVICE)
criterion = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model_w6.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25, eta_min=1e-5)

N_EPOCHS = 25
history = {"train_loss":[], "val_loss":[], "train_f1":[], "val_f1":[]}
best_val_f1, best_epoch = 0, 0

print(f"Training ImprovedCNN — {N_EPOCHS} epochs, CosineAnnealingLR")
print(f"{'Ep':>4} {'TrL':>8} {'VaL':>8} {'TrF1':>7} {'VaF1':>7}")
print("─"*40)
for ep in range(1, N_EPOCHS+1):
    tr_l, tr_f = train_epoch(model_w6, train_loader, criterion, opt, DEVICE)
    va_l, va_f = eval_epoch(model_w6, val_loader, criterion, DEVICE)
    scheduler.step()
    for k,v in zip(["train_loss","val_loss","train_f1","val_f1"],[tr_l,va_l,tr_f,va_f]):
        history[k].append(v)
    if va_f > best_val_f1:
        best_val_f1 = va_f; best_epoch = ep
        torch.save(model_w6.state_dict(), "models/improved_cnn_best.pth")
    print(f"{ep:>4} {tr_l:>8.4f} {va_l:>8.4f} {tr_f:>7.3f} {va_f:>7.3f}")

print(f"\nBest val F1: {best_val_f1:.3f} at epoch {best_epoch}")
print(f"Fill in val_f1={best_val_f1:.3f} in your experiment log above.")

# Evaluate per-city on best checkpoint
model_w6.load_state_dict(torch.load("models/improved_cnn_best.pth", map_location=DEVICE))
city_f1 = per_city_f1(model_w6, df, DEVICE)
print(f"\nPer-city F1:")
for city, f1 in sorted(city_f1.items(), key=lambda x: -x[1]):
    print(f"  {city:<8}: {f1:.3f}  {'█'*int(f1*30)}")
print(f"Equity gap: {max(city_f1.values())-min(city_f1.values()):.3f}")

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Changing 2+ hyperparameters at once | One at a time — record all |
| Not computing per-city F1 | Add to every training run |
| CosineAnnealingLR T_max too short | Set T_max >= expected epochs |

## Week 6 Complete

Next: `week7_resnet_STARTER.ipynb`

*The Computer Vision Internship · VisionAI Lagos*

## ✅ By completing Week 6, you can now:

- Apply dropout, batch normalisation, and LR scheduling to improve the CNN
- Log all experiments with architecture, hyperparameters, and val accuracy
- Explain why changing multiple things simultaneously makes debugging impossible
- Produce a ranked experiment table showing the best configuration

📤 **GitHub:** Push week6_improving_cnn_STARTER.ipynb and experiment_log.csv. Commit: "Week 6: CNN improved, experiments documented"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
